In [64]:
import kagglehub


path = kagglehub.dataset_download("tobiasbueck/multilingual-customer-support-tickets")

print("Path to dataset files:", path)


Path to dataset files: /home/hello/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14


In [87]:
import pandas as pd 

df = pd.read_csv("data/aa_dataset-tickets-multi-lang-5-2-50-version.csv")
df.head(2)

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN


In [88]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28587 entries, 0 to 28586
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   subject   24749 non-null  str  
 1   body      28587 non-null  str  
 2   answer    28580 non-null  str  
 3   type      28587 non-null  str  
 4   queue     28587 non-null  str  
 5   priority  28587 non-null  str  
 6   language  28587 non-null  str  
 7   version   28587 non-null  int64
 8   tag_1     28587 non-null  str  
 9   tag_2     28574 non-null  str  
 10  tag_3     28451 non-null  str  
 11  tag_4     25529 non-null  str  
 12  tag_5     14545 non-null  str  
 13  tag_6     5874 non-null   str  
 14  tag_7     2040 non-null   str  
 15  tag_8     565 non-null    str  
dtypes: int64(1), str(15)
memory usage: 3.5 MB


In [89]:
df.shape

(28587, 16)

In [90]:
df["language"].value_counts()

language
en    16338
de    12249
Name: count, dtype: int64

In [91]:
df.isnull().sum()

subject      3838
body            0
answer          7
type            0
queue           0
priority        0
language        0
version         0
tag_1           0
tag_2          13
tag_3         136
tag_4        3058
tag_5       14042
tag_6       22713
tag_7       26547
tag_8       28022
dtype: int64

In [92]:
df["subject"] = df["subject"].fillna("")

In [93]:
df = df.dropna(subset=['body'])

In [96]:
df = df[df["language"]== "en"]

In [97]:
df["text"] = df["subject"]+" "+df["body"]

In [98]:
df.isnull().sum()

subject         0
body            0
answer          3
type            0
queue           0
priority        0
language        0
version         0
tag_1           0
tag_2           6
tag_3          69
tag_4        1711
tag_5        7922
tag_6       12968
tag_7       15204
tag_8       16057
text            0
dtype: int64

In [99]:
df = df[["text", 'queue','type']] 
df

,text,queue,type
1,"Account Disruption Dear Customer Support Team,...",Technical Support,Incident
2,Query About Smart Home System Integration Feat...,Returns and Exchanges,Request
3,Inquiry Regarding Invoice Details Dear Custome...,Billing and Payments,Request
4,Question About Marketing Agency Software Compa...,Sales and Pre-Sales,Problem
5,"Feature Query Dear Customer Support,\n\nI hope...",Technical Support,Request
...,...,...,...
28578,Problem with Billing Adjustment An unexpected ...,Billing and Payments,Incident
28580,Urgent: Incident Involving Data Breach in Medi...,Product Support,Problem
28582,Performance Problem with Data Analytics Tool T...,Technical Support,Incident
28585,Update Request for SaaS Platform Integration F...,IT Support,Change


In [100]:
df.rename(
    columns = {
        'queue':'category',
        'type':"issuse_type"
    },inplace = True
)

In [101]:
df

,text,category,issuse_type
1,"Account Disruption Dear Customer Support Team,...",Technical Support,Incident
2,Query About Smart Home System Integration Feat...,Returns and Exchanges,Request
3,Inquiry Regarding Invoice Details Dear Custome...,Billing and Payments,Request
4,Question About Marketing Agency Software Compa...,Sales and Pre-Sales,Problem
5,"Feature Query Dear Customer Support,\n\nI hope...",Technical Support,Request
...,...,...,...
28578,Problem with Billing Adjustment An unexpected ...,Billing and Payments,Incident
28580,Urgent: Incident Involving Data Breach in Medi...,Product Support,Problem
28582,Performance Problem with Data Analytics Tool T...,Technical Support,Incident
28585,Update Request for SaaS Platform Integration F...,IT Support,Change


In [102]:
df['category'].value_counts()

category
Technical Support                  4737
Product Support                    3073
Customer Service                   2410
IT Support                         1942
Billing and Payments               1595
Returns and Exchanges               820
Service Outages and Maintenance     664
Sales and Pre-Sales                 513
Human Resources                     348
General Inquiry                     236
Name: count, dtype: int64

It was little bit imbalance , lets deal with it later

In [103]:
df.isnull().sum()

text           0
category       0
issuse_type    0
dtype: int64

In [104]:
df['text_length'] = df['text'].apply(len)
df['text_length'].mean()

np.float64(405.5386828253152)

In [105]:
df['text_length'] = df['text'].apply(lambda x: len(x.split()))
print(df['text_length'].mean())

58.64304076386339


In [106]:
df['text'].sample(5)

4156     Revise Support for Integration Updates Draftin...
23195    SAP Data Security Customer Support, seeking gu...
4759     Potential Data Breach at Healthcare Provider A...
11229    Problem with System Down Today Dear Support Te...
7862     Examining Delays in Loading Project Dashboards...
Name: text, dtype: str

In [107]:
from collections import Counter
all_words = " ".join(df['text']).lower().split()
Counter(all_words).most_common(40)

[('the', 42354),
 ('to', 37197),
 ('and', 30479),
 ('data', 12445),
 ('in', 12123),
 ('for', 12065),
 ('i', 11260),
 ('a', 10618),
 ('you', 10553),
 ('on', 10033),
 ('with', 9359),
 ('we', 9113),
 ('this', 8913),
 ('would', 8286),
 ('our', 7687),
 ('your', 7586),
 ('of', 7470),
 ('is', 7408),
 ('have', 7217),
 ('be', 7020),
 ('could', 6733),
 ('issue', 6047),
 ('customer', 5835),
 ('security', 5804),
 ('are', 5648),
 ('assistance', 5267),
 ('provide', 5238),
 ('appreciate', 5224),
 ('am', 4963),
 ('that', 4946),
 ('software', 4904),
 ('due', 4800),
 ('as', 4751),
 ('digital', 4722),
 ('medical', 4709),
 ('analytics', 4623),
 ('investment', 4549),
 ('support', 4338),
 ('please', 4321),
 ('about', 4299)]

In [108]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re

all_stop_words = ENGLISH_STOP_WORDS.union(['hi', 'hello', 'thanks', 'thank', 'regards', 'dear'])


In [127]:
def remove_stopwords(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]"," ",text)
    words = text.split()
    words = [w for w in words if w not in all_stop_words and len(w) > 2]
    return " ".join(words)

df["clean_text"] = df['text'].apply(remove_stopwords)
df['clean_text']
    

1        account disruption writing report significant ...
2        query smart home integration features hope mes...
3        inquiry regarding invoice details hope message...
4        question marketing agency software compatibili...
5        feature query hope message reaches good health...
                               ...                        
28578    billing adjustment unexpected billing discrepa...
28580    urgent incident involving breach medical recor...
28582    performance analytics tool analytics tool expe...
28585    update saas platform integration features requ...
28586    inquiry project management features detailed p...
Name: clean_text, Length: 16338, dtype: str

In [119]:
df

,text,category,issuse_type,text_length,clean_text
1,"Account Disruption Dear Customer Support Team,...",Technical Support,Incident,84,account disruption customer support team n ni ...
2,Query About Smart Home System Integration Feat...,Returns and Exchanges,Request,83,query smart home integration features customer...
3,Inquiry Regarding Invoice Details Dear Custome...,Billing and Payments,Request,95,inquiry regarding invoice details customer sup...
4,Question About Marketing Agency Software Compa...,Sales and Pre-Sales,Problem,103,question marketing agency software compatibili...
5,"Feature Query Dear Customer Support,\n\nI hope...",Technical Support,Request,99,feature query customer support n ni hope messa...
...,...,...,...,...,...
28578,Problem with Billing Adjustment An unexpected ...,Billing and Payments,Incident,21,problem billing adjustment unexpected billing ...
28580,Urgent: Incident Involving Data Breach in Medi...,Product Support,Problem,21,urgent incident involving data breach medical ...
28582,Performance Problem with Data Analytics Tool T...,Technical Support,Incident,16,performance problem data analytics tool data a...
28585,Update Request for SaaS Platform Integration F...,IT Support,Change,48,update request saas platform integration featu...


ok... Lets find Keywords belonging to the category

In [ ]:
def find_keywords():
    for item in df['category'].unique():
        print(f"Category : {item}")
        subset = df[df['category']== item]
        words = " ".join(subset['clean_text']).lower().split()
        common = Counter(words).most_common(60)
        print(common)
        # print("\n")

find_keywords()

There are many corpus-level stopwords. 

In [123]:
custom_stopwords = set([
    'data','support','issue','issues','information','provide','request',
    'assistance','customer','appreciate','regards','thanks','thank',
    'dear','team','help','looking','forward','greatly','soon','problem','greatly','great','problems'
])

all_stop_words = ENGLISH_STOP_WORDS.union(custom_stopwords)

In [124]:
df["clean_text"] = df['text'].apply(remove_stopwords)
df['clean_text']
    

1        account disruption n ni writing report signifi...
2        query smart home integration features n ni hop...
3        inquiry regarding invoice details n ni hope me...
4        question marketing agency software compatibili...
5        feature query n ni hope message reaches good h...
                               ...                        
28578    billing adjustment unexpected billing discrepa...
28580    urgent incident involving breach medical recor...
28582    performance analytics tool analytics tool expe...
28585    update saas platform integration features requ...
28586    inquiry project management features detailed p...
Name: clean_text, Length: 16338, dtype: str

In [128]:
find_keywords()

Category : Technical Support
[('software', 2206), ('security', 1862), ('analytics', 1442), ('medical', 1367), ('integration', 1327), ('resolve', 1260), ('digital', 1169), ('investment', 1152), ('recent', 1107), ('marketing', 1105), ('project', 1085), ('updates', 1047), ('tools', 1036), ('matter', 1014), ('access', 1010), ('platform', 1001), ('systems', 945), ('management', 918), ('saas', 861), ('restarting', 860), ('guidance', 845), ('breach', 793), ('performance', 786), ('persists', 730), ('despite', 727), ('steps', 721), ('hospital', 717), ('network', 697), ('server', 693), ('possible', 672), ('strategies', 664), ('devices', 654), ('look', 621), ('resolving', 621), ('solution', 602), ('efforts', 573), ('need', 572), ('facing', 559), ('potential', 558), ('brand', 538), ('know', 524), ('attempts', 488), ('measures', 485), ('updating', 485), ('cause', 483), ('update', 480), ('troubleshooting', 468), ('compatibility', 458), ('impacting', 458), ('integrating', 448), ('incident', 448), ('r

In [129]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features = 5000)

x = vectorizer.fit_transform(df['clean_text'])
y = df['category']
print("{x} ----- {y}")

{x} ----- {y}


In [130]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [131]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 200)
model.fit(x_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [133]:
from sklearn.metrics import classification_report

y_pred = model.predict(x_test)

print(classification_report(y_test, y_pred))

                                 precision    recall  f1-score   support

           Billing and Payments       0.86      0.67      0.75       301
               Customer Service       0.38      0.35      0.37       496
                General Inquiry       1.00      0.05      0.10        37
                Human Resources       1.00      0.12      0.21        68
                     IT Support       0.43      0.23      0.30       362
                Product Support       0.38      0.41      0.40       620
          Returns and Exchanges       0.79      0.11      0.19       175
            Sales and Pre-Sales       0.79      0.09      0.16       127
Service Outages and Maintenance       0.84      0.54      0.66       161
              Technical Support       0.44      0.75      0.56       921

                       accuracy                           0.47      3268
                      macro avg       0.69      0.33      0.37      3268
                   weighted avg       0.53      0

In [136]:
import os
os.mkdir('models')

In [137]:
import pickle
pickle.dump(model, open("models/category_model.pkl",'wb'))
pickle.dump(vectorizer,open('models/vectorizer.pkl','wb'))